# B 날씨 사건 · 연구 기간 · 악천후일 목록

담당: B · 결과물은 **Train + Dev(`train_dev`) / Test(`holdout`)** 로 나눠 저장한다. Train과 Dev는 `model_set` 열로 고른다.

1. ASOS 받기: 터미널에서 `python crawling/kma_asos.py` (api_keys.txt 에 `DATA_GO_KR_KEY=` 필요)
   - API가 안 되면 기상자료개방포털에서 받은 CSV를 `data/raw/asos_manual/` 에 넣어도 된다.
2. 기상특보 받기: `python crawling/kma_warning.py` (크롬이 열려 자동으로 조회 · 페이지 넘김 · 상세 수집)
3. 표 만들기: 터미널에서 `python src/clean/weather.py`. 이 노트북은 같은 계산을 단계별로 보여 준다. 마지막 저장 셀은 처음 셀의 `RUN_BUILD = True`일 때만 저장한다 (기본 `False`라 다시 실행해도 공통 표를 덮어쓰지 않는다).

| 결과 | 위치 |
|---|---|
| 시간별 날씨 | `data/shared/train_dev/weather_hourly_train_dev.csv`, `data/shared/holdout/weather_hourly_holdout.csv` |
| 눈 · 비 사건 | `data/shared/{train_dev,holdout}/weather_events_*.csv` |
| 악천후일 · 비교일 (C 교통량 수집용) | `data/shared/{train_dev,holdout}/weather_days_*.csv` |
| 시즌별 첫 · 마지막 날, 연구 기간 후보 | `outputs/tables/season_first_last.csv`, `outputs/tables/research_period_options.csv` |
| 서울 특보 구간 (생존분석 출발점) | `data/shared/{train_dev,holdout}/weather_warnings_*.csv` |
| 연구 기간 · 기준값 (팀 공유) | `config/periods.yaml` |

In [ ]:
import sys
from pathlib import Path
def find_project_root(start=None):
    """지금 폴더에서 위로 올라가며 README.md와 .git이 있는 저장소 루트를 찾는다."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'README.md').is_file() and (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

ROOT = find_project_root()
RUN_BUILD = False  # True로 바꿀 때만 마지막 셀에서 공통 표 · periods.yaml을 저장한다 (기본은 계산 · 그림만)
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.clean import weather as W

plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우 한글 글꼴
plt.rcParams['axes.unicode_minus'] = False
W.THRESHOLDS

## 1. 읽기 · 시즌 구분

In [ ]:
raw = W.add_season(W.load_asos())
print(raw['datetime'].min(), '~', raw['datetime'].max(), len(raw), '시간')
print('관측 빠진 시간:', int(raw['missing_obs'].sum()))
raw.groupby(['split', 'season']).size()

## 2. 눈 · 비 사건

In [ ]:
hourly, ev = W.extract_events(raw)
ev.groupby(['split', 'season', 'intensity']).size().unstack(fill_value=0)

## 3. 시즌별 첫 · 마지막 날

겨울은 **쌓인 눈(신적설 0.1cm 이상)** 기준, 여름은 **하루 30mm 이상 큰비 날** 기준이다. 안 쌓인 눈까지 포함한 첫 · 마지막 눈도 참고로 적었다.

In [ ]:
fl = W.first_last(hourly, ev)
fl[['name', 'split', 'kind', 'first', 'last', 'n_events', 'first_any_snow', 'last_any_snow']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, k, title in ((axes[0], 'W', '겨울: 첫 · 마지막 쌓인 눈 (10/1부터 며칠째)'), (axes[1], 'S', '여름: 첫 · 마지막 큰비 날 (5/1부터 며칠째)')):
    t = fl[fl['season'].str.startswith(k)]
    for i, r in enumerate(t.itertuples()):
        c = 'tab:blue' if r.split == 'train' else 'tab:orange'
        ax.plot([r.first_offset, r.last_offset], [i, i], color=c, lw=6, solid_capstyle='butt')
    ax.set_yticks(range(len(t)), t['name'])
    ax.set_title(title)
    ax.invert_yaxis()
axes[0].plot([], [], color='tab:blue', lw=6, label='학습')
axes[0].plot([], [], color='tab:orange', lw=6, label='평가')
axes[0].legend()
plt.tight_layout()
plt.savefig(ROOT / 'outputs/figures/season_first_last.png', dpi=150)

## 4. 연구 기간 후보

학습 3년의 첫 · 마지막 날로 후보를 만든다. `train_coverage` 는 학습 기간 악천후 날(겨울: 쌓인 눈 사건, 여름: 큰비 날) 중 그 기간 안에 들어오는 비율이다.

**추천 규칙**: 학습 coverage 90% 이상인 후보 중 가장 짧은 것. 평가 시즌은 기간을 정할 때 쓰지 않고, 들어오는지만 확인한다.

In [ ]:
opts = W.period_options(hourly, ev, fl)
rec = W.recommend_period(opts, min_coverage=0.9)
display(opts.style.format({'train_coverage': '{:.0%}', 'eval_coverage': '{:.0%}'}))
rec

## 5. 악천후일 · 비교일 (C 교통량 수집용)

교통량 API는 한 번에 1지점 × 1시간만 주므로 날짜 수를 제한한다 (학습 30일 · 평가 15일, 2025-12-04는 항상 포함).

In [ ]:
days, hourly = W.select_days(hourly, ev, rec, max_train=30, max_eval=15)
print(days.groupby(['split', 'role']).size())
days.head(20)

## 6. 서울 기상특보 상태 (crawling/kma_warning.py 결과)

시간마다 `warn_snow · warn_rain · warn_cold` (0 없음 · 1 주의보 · 2 경보), 발효 권역 수, **발표됐지만 아직 발효 전(upcoming)**, 발효 후 지난 시간을 붙인다. 그 시각까지 발효된 것만 반영해 미래 정보가 섞이지 않는다.

In [ ]:
hourly, w_items, w_eps = W.add_warnings(hourly)
if w_eps is not None:
    display(w_eps.groupby(['split', 'kind']).size().unstack(fill_value=0))
    display(hourly.loc[(hourly['datetime'] >= '2025-12-04 14:00') & (hourly['datetime'] <= '2025-12-05 10:00'),
                       ['datetime', 'precip_type', 'warn_snow', 'warn_snow_areas', 'warn_snow_upcoming', 'warn_snow_hours']])

## 7. 저장 (`RUN_BUILD = True`일 때만 · train_dev / holdout)

In [ ]:
if not RUN_BUILD:
    print('건너뜀: RUN_BUILD = False (data/shared · outputs/tables · config/periods.yaml 저장 없음)')
else:
    (ROOT / 'outputs/figures').mkdir(parents=True, exist_ok=True)
    W.save_outputs(hourly, ev, fl, opts, rec, days, warn_items=w_items, warn_eps=w_eps)
print((ROOT / 'config/periods.yaml').read_text(encoding='utf-8')[:800])